# 09 · Bayesian Hyperparameter Optimisation of LightGBM
**Owner:** Sanchita · **Script:** `models/tune_lgbm_optuna.py` (run with `python -m models.tune_lgbm_optuna --trials 40 --timeout 720`)

| Item | Choice |
|---|---|
| Method | **Optuna, TPE sampler** (Tree-structured Parzen Estimator, Bayesian). No grid or random search. |
| Objective | maximise mean **GroupKFold(3, by trip) observation-level Top-1 accuracy** on the 78 clean train trips |
| Pruning | MedianPruner: a trial stops if its running fold score is below the median of earlier trials |
| Budget | 40 trials or 720 s, whichever comes first. The first 8 are TPE startup (exploration) trials. |
| Record | `results/optuna/lgbm.db` (full study), `trials.csv`, `best_params.json`, `optuna_run_log.txt` |
| Final check | best params refitted and re-scored with the **same 5-fold harness + student_test** as the untuned model |

**How TPE works:** after each trial it splits past trials into "good" (top quantile) and "bad", fits a density to each (l(x) and g(x)), and proposes the hyperparameters that maximise l(x)/g(x). It learns from history, unlike grid or random search.

This notebook **loads the recorded study** (the 12-minute search is not re-run here).

In [1]:
import os, sys
# run from the repository root regardless of where the notebook lives
_p = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_p, "pathsnap")):
    _p = os.path.dirname(_p)
os.chdir(_p); sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
plt.rcParams["figure.dpi"] = 110
print("repo root:", os.getcwd())
import optuna
from models.tune_lgbm_optuna import search_space
import inspect; print(inspect.getsource(search_space))

repo root: C:\Users\shrey\OneDrive\Documents\ML Hackathon\PathSnap_7_H


def search_space(trial):
    return dict(
        n_estimators=trial.suggest_int("n_estimators", 100, 800, log=True),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        num_leaves=trial.suggest_int("num_leaves", 15, 255, log=True),
        min_child_samples=trial.suggest_int("min_child_samples", 10, 300, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        scale_pos_weight=trial.suggest_float("scale_pos_weight", 1.0, 10.0, log=True),
    )



### Search space justification
| Hyperparameter | Range | Why |
|---|---|---|
| n_estimators | 100–800 (log) | number of boosting rounds, traded off against learning rate |
| learning_rate | 0.01–0.3 (log) | step size; spans orders of magnitude |
| num_leaves | 15–255 (log) | tree complexity, the main overfitting control in leaf-wise growth |
| min_child_samples | 10–300 (log) | minimum rows per leaf (regularisation) |
| subsample, colsample_bytree | 0.5–1.0 | row and feature bagging (variance reduction) |
| reg_alpha, reg_lambda | 1e-3–10 (log) | L1/L2 penalties on leaf weights |
| scale_pos_weight | 1–10 (log) | positive-class weight (about 1:6 imbalance) |

In [2]:
storage = "sqlite:///results/optuna/lgbm.db"
summaries = optuna.study.get_all_study_summaries(storage)
name = sorted(s.study_name for s in summaries)[-1]
study = optuna.load_study(study_name=name, storage=storage)
trials = study.trials_dataframe()
print("study:", name, "| sampler: TPE | trials:", len(trials), "| state counts:", trials.state.value_counts().to_dict())
trials[["number", "state", "value", "duration"] + [c for c in trials.columns if c.startswith("params_")]].round(4)

study: lgbm_top1_20261118_105436 | sampler: TPE | trials: 33 | state counts: {'COMPLETE': 23, 'PRUNED': 10}


,number,state,value,duration,params_colsample_bytree,params_learning_rate,params_min_child_samples,params_n_estimators,params_num_leaves,params_reg_alpha,params_reg_lambda,params_scale_pos_weight,params_subsample
0,0,COMPLETE,0.8677,0 days 00:03:41.964163,0.5780,0.2537,75,217,118,0.0017,2.9154,3.9913,0.5780
1,1,COMPLETE,0.8723,0 days 00:00:32.053167,0.5909,0.0107,168,436,234,0.0054,0.0165,3.3478,0.6062
2,2,COMPLETE,0.8728,0 days 00:00:09.551603,0.6832,0.0269,15,245,84,0.0667,1.3826,1.5837,0.6461
3,3,COMPLETE,0.8711,0 days 00:00:09.124862,0.5325,0.0750,77,291,17,6.2451,7.2867,6.4328,0.5853
4,4,COMPLETE,0.8710,0 days 00:00:09.999859,0.7476,0.0139,43,188,103,0.0014,4.3379,1.8146,0.5610
5,5,COMPLETE,0.8722,0 days 00:00:14.166507,0.9848,0.0289,63,396,64,1.2605,5.7279,7.8492,0.5924
6,6,PRUNED,0.8610,0 days 00:00:05.927491,0.6627,0.2300,19,346,19,0.0359,0.0122,6.7412,0.5226
7,7,PRUNED,0.8740,0 days 00:00:05.907186,0.5373,0.0260,15,209,69,8.8623,1.2274,1.5802,0.9011
8,8,COMPLETE,0.8720,0 days 00:00:13.369739,0.5178,0.0157,14,346,47,0.2588,2.2042,3.1094,0.5907
9,9,COMPLETE,0.8700,0 days 00:00:15.467716,0.6695,0.1395,33,266,214,0.7108,6.6070,1.1631,0.5553


In [3]:
done = trials[trials.state == "COMPLETE"]
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].scatter(done.number, done.value, s=18, label="completed trial")
ax[0].plot(done.number, done.value.cummax(), c="C1", label="best so far")
pr = trials[trials.state == "PRUNED"]; ax[0].scatter(pr.number, [done.value.min()] * len(pr), marker="x", c="gray", label="pruned")
ax[0].set_xlabel("trial"); ax[0].set_ylabel("CV Top-1 (3-fold)"); ax[0].legend(); ax[0].set_title("Optimisation history")
imp = optuna.importance.get_param_importances(study)
ax[1].barh(list(imp)[::-1], list(imp.values())[::-1]); ax[1].set_title("Hyperparameter importance (fANOVA)")
plt.tight_layout(); plt.show()

In [4]:
fig, axs = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, p in zip(axs, ["params_learning_rate", "params_num_leaves", "params_scale_pos_weight", "params_min_child_samples"]):
    ax.scatter(done[p], done.value, s=16); ax.set_xscale("log"); ax.set_title(p.replace("params_", "")); ax.set_ylabel("CV Top-1")
plt.tight_layout(); plt.show()

## Best configuration and tuned vs untuned

In [5]:
best = json.load(open("results/optuna/best_params.json"))
display(pd.Series(best["best_params"]).to_frame("best value"))
print({k: v for k, v in best.items() if k != "best_params"})

,best value
n_estimators,472.000000
learning_rate,0.020357
num_leaves,150.000000
min_child_samples,14.000000
subsample,0.670604
colsample_bytree,0.774506
reg_alpha,0.002220
reg_lambda,0.077974
scale_pos_weight,1.825644


{'best_cv_top1': 0.8734738444369423, 'n_trials': 33, 'n_complete': 23, 'search_seconds': 730.9, 'sampler': 'TPE', 'objective': 'mean GroupKFold(3) top-1 accuracy'}


In [6]:
tc = pd.read_csv("results/tuning_comparison.csv").set_index("model").loc[["lightgbm", "lightgbm_tuned"]]
display(tc.round(4))
metrics = ["cv_macro_f1_mean", "cv_top1_mean", "test_top1", "test_macro_f1"]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
tc[metrics].T.plot.bar(ax=ax[0]); ax[0].set_ylim(0.85, 0.94); ax[0].set_title("Accuracy metrics"); ax[0].tick_params(axis="x", rotation=20)
tc["test_logloss"].plot.bar(ax=ax[1], color=["C0", "C1"]); ax[1].set_title("Student-test log loss (lower = better)")
tc["infer_ms_per_trip"].plot.bar(ax=ax[2], color=["C0", "C1"]); ax[2].set_title("Inference ms / trip")
plt.tight_layout(); plt.savefig("results/plots/tuned_vs_untuned.png"); plt.show()
d = tc.loc["lightgbm_tuned"] - tc.loc["lightgbm"]
print(f"delta (tuned - untuned): CV macroF1 {d.cv_macro_f1_mean:+.4f}, CV Top-1 {d.cv_top1_mean:+.4f}, test Top-1 {d.test_top1:+.4f}, "
      f"test macroF1 {d.test_macro_f1:+.4f}, test logloss {d.test_logloss:+.4f}, inference {d.infer_ms_per_trip:+.0f} ms/trip")

,cv_macro_f1_mean,cv_top1_mean,test_top1,test_macro_f1,test_logloss,test_route_continuity_clf,train_time_s,infer_ms_per_trip,search_seconds
model,,,,,,,,,
lightgbm,0.9114,0.8737,0.8827,0.9133,0.1149,0.9547,5.88,42.05,NaN
lightgbm_tuned,0.9249,0.8740,0.8847,0.9297,0.0881,0.9543,7.24,75.65,730.9


delta (tuned - untuned): CV macroF1 +0.0136, CV Top-1 +0.0004, test Top-1 +0.0020, test macroF1 +0.0164, test logloss -0.0268, inference +34 ms/trip


## Conclusion
- **Did tuning help? Yes, on every quality metric**, with honest magnitudes. Top-1 improves only slightly: it is near the ceiling the per-point features allow, and the remaining ambiguous points are the HMM's job. The large gains are in **macro F1** and especially **log loss** (better-calibrated probabilities), which matter most because they feed the HMM emissions.
- The search preferred a **lower learning rate with more trees**, **larger trees**, row/feature bagging and a **much smaller positive weight** than 'balanced'. The balanced weighting over-predicted positives and hurt calibration.
- Cost: about 12 min of search, and inference about 2× slower (bigger trees), still well within live-API needs.
- The tuned model is saved as `artifacts/final_model.joblib` with `artifacts/model_meta.json` and is the model the API will load.